# ReBRAC Paper-Readiness Follow-Up — 4 项必做合集

> 文档对应：[docs/rebrac_mainline_review.md §3.1](../docs/rebrac_mainline_review.md)
> 目的：在 paper drafting 之前，用 ≤ 1 day L4 + ≤ 2h 写作堵掉 4 个最显眼的审稿人弱点。
> 收口要求：跑完所有 cell 后，按 §6 写入清单更新 plan / report / paper draft。

## 执行摘要

| # | 任务 | 类型 | 预算 | 目的 |
|---|---|---|---|---|
| **A** | Phase 2 seed 45/46 补点（升 5-seed） | 训练 | ~2h L4 | 可正式做 std 对比（堵 review §2.2.2） |
| **B** | critic LayerNorm-off probe（crosscomp-1000, 2 seeds） | 训练 | ~2h L4 | 证明 dual penalty 不是 LayerNorm 的伪装（堵 review §2.2.4） |
| **C** | ReBRAC dep vs TD3BC priv 统计检验 | 分析 | ~30 min | 量化 "持平" 的 statistical claim（堵 review §2.2.3） |
| **D** | Q-normalized 变体 method section 草稿 | 写作 | ~1h | 显式声明 actor loss 实现（堵 review §2.2.1） |

A + B 加起来 ~4h L4，C + D 加起来 ~1.5h 笔头时间，全部完成后即可启动 paper drafting。

## 与现有实验输出树的隔离

| 任务 | 输出根（与已有 stage 不冲突） |
|---|---|
| A | `checkpoints/offline/rebrac/worldcomp_teacher_gap/privileged_critic/`（Phase 2 同根，靠 driver skip 处理已存在 seeds 42/43/44）|
| B | `checkpoints/offline/rebrac/critic_ln_off/`、`results/offline/rebrac/critic_ln_off/` |
| C | `docs/rebrac_statistical_test_followup.md`（纯分析产物） |
| D | `docs/rebrac_method_section_draft.md`（纯写作产物） |


## 0. 环境 sanity check

In [ ]:
!lscpu | head -10
print()
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

## 1. 通用配置

A / B 两个训练任务共用的 base env vars。各任务再自己 override。

In [ ]:
import os

os.environ["PYTHON_BIN"]         = "python3"
os.environ["DEVICE"]             = "cuda"
os.environ["EVAL_WORKERS"]       = "6"
os.environ["EVAL_WORKER_DEVICE"] = "cpu"


## 2. 任务 A — Phase 2 seed 45/46 补点（升到 5-seed）

### 2.1 scope / judgement

**目的**：把 worldcomp privileged-critic Phase 2（[plan §6.5.3](../docs/rebrac_experiment_plan.md)、[report §7.12](../docs/rebrac_experiment_report.md)）从 3 seeds 升到 5 seeds，以便：

1. 可以与 TD3BC privileged-critic formal（5 seeds、std=0.086）做严格 std 对比；
2. 移除 review §2.2.2 提到的 "3 seeds 含事先选择的 seed 44 → cherry-pick" 风险。

**实验 scope**：

| 轴 | 配置 | 说明 |
|---|---|---|
| β1 / β2 | `4.0 / 2.0` | Stage C 锁定的 finalist |
| dataset | `worldcomp-1000` | 与 Phase 1 / Phase 2 同 dataset |
| seeds | `42 43 44 45 46` | driver skip 已有的 42/43/44；只新跑 45/46 |
| 轨道 | privileged-critic | actor=deployable obs，critic=privileged obs（`zeros` 模式）|
| TRAIN_EPOCHS | `64` | 与 Phase 1 / Phase 2 一致 |
| val / test manifest | 40 / 100 | 复用 `benchmarks/offline_rebrac_worldcomp_final/` |

**预算**：仅 2 个新 train run（45 / 46），约 2h L4。

**判定**：

| 条件 | 阈值 | 解读 |
|---|---|---|
| 5-seed mean | > 0.922（TD3BC priv 5-seed mean） | "ReBRAC priv 与 TD3BC priv 持平" 在 5-seed 下仍成立 |
| 5-seed std | ≤ 0.10 | 与 Phase 1 deployable 同口径 |
| Δ vs 3-seed mean | 在 ±2pp 内 | 3-seed 估计未明显偏 |
| Δ vs 3-seed std | std 上升幅度合理（3→5 seeds 估计放宽是预期的）| 不可声明 std 严格 < TD3BC priv，除非 5-seed std 仍显著低于 0.086 |


### 2.2 driver 配置 + 跑 privileged_train → validate → test → summarize

In [ ]:
# —— Stage C 锁定的 finalist ——
os.environ["ACTOR_PENALTY_COEF"]  = "4.0"
os.environ["CRITIC_PENALTY_COEF"] = "2.0"

# —— Phase 2 升到 5-seed：driver 会 skip 42/43/44（trainer_state.json + agent_final.pt 存在）——
os.environ["PRIVILEGED_FINAL_SEEDS"]                       = "42 43 44 45 46"
os.environ["PRIVILEGED_FINAL_TRAIN_EPOCHS"]                = "64"
os.environ["PRIVILEGED_FINAL_CHECKPOINT_EVERY_EPOCHS"]     = "8"
os.environ["PRIVILEGED_ACTOR_UPDATE_MODE"]                 = "zeros"
os.environ["FINAL_VAL_MANIFEST_EPISODES"]                  = "40"
os.environ["FINAL_TEST_MANIFEST_EPISODES"]                 = "100"

# 其余继承 driver 默认（DATASET_POLICY=worldcomp, DATASET_EPISODES_VALUE=1000，
# BENCHMARK_KEY=single_u10_cross_tgt15, PROBE_LAYOUT=s0, etc.）


In [ ]:
# 5-seed 全流程；driver 对已存在的 42/43/44 自动 skip，只训 45/46
for mode in ["privileged_train", "privileged_validate", "privileged_test", "privileged_summarize"]:
    os.environ["MODE"] = mode
    !bash scripts/run_offline_rebrac_worldcomp_teacher_gap.sh


### 2.3 5-seed 主结果 + per-seed 分布

In [ ]:
import json
from pathlib import Path

import pandas as pd

A_RESULTS_ROOT = Path("results/offline/rebrac/worldcomp_teacher_gap/privileged_critic")
A_DATASET     = "worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000"
A_PAIR        = "actorb_4p0__criticb_2p0"
A_SEEDS       = os.environ["PRIVILEGED_FINAL_SEEDS"].split()


def load_test_per_seed(root: Path, dataset: str, pair: str, seeds: list[str]) -> pd.DataFrame:
    rows = []
    for seed in seeds:
        path = root / dataset / pair / "test" / f"seed_{seed}.json"
        if not path.exists():
            print(f"[warn] missing: {path}")
            continue
        d = json.loads(path.read_text(encoding="utf-8"))
        rows.append({
            "seed": seed,
            "success_rate": d["eval_success_rate"],
            "return": d["eval_return"],
            "safety_cost": d["eval_safety_cost"],
            "time_s": d["eval_time_s"],
            "path_efficiency": d.get("eval_path_efficiency"),
        })
    return pd.DataFrame(rows)


a_per_seed = load_test_per_seed(A_RESULTS_ROOT, A_DATASET, A_PAIR, A_SEEDS)
print("[per-seed Phase 2 — 5 seeds × test=100 (privileged-critic)]")
print(a_per_seed.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

if not a_per_seed.empty:
    print()
    print("[5-seed summary]")
    a_mean = a_per_seed["success_rate"].mean()
    a_std  = a_per_seed["success_rate"].std()
    print(f"  mean test success_rate = {a_mean:.4f}")
    print(f"  std  test success_rate = {a_std:.4f}")


### 2.4 vs 3-seed 旧结果 / vs TD3BC priv 5-seed 严格 std 对比

In [ ]:
# 3-seed 旧基线（Phase 2 rev.5 时保留）
A_3SEED_MEAN = 0.9267
A_3SEED_STD  = 0.025

# TD3BC privileged-critic formal 5-seed
TD3BC_PRIV_MEAN = 0.922
TD3BC_PRIV_STD  = 0.086

if not a_per_seed.empty:
    print("=" * 80)
    print(f"{'Protocol':<54}{'mean':>8}{'std':>8}{'n':>4}")
    print("-" * 80)
    print(f"{'TD3BC privileged-critic formal (5s)':<54}{TD3BC_PRIV_MEAN:>8.3f}{TD3BC_PRIV_STD:>8.3f}{5:>4}")
    print(f"{'ReBRAC priv Phase 2 (rev.5, 3 seeds 42/43/44)':<54}{A_3SEED_MEAN:>8.4f}{A_3SEED_STD:>8.4f}{3:>4}")
    print(f"{'ReBRAC priv Phase 2 (5 seeds, this notebook)':<54}{a_mean:>8.4f}{a_std:>8.4f}{5:>4}")
    print("=" * 80)

    delta_mean_vs_3seed = a_mean - A_3SEED_MEAN
    delta_std_vs_3seed  = a_std - A_3SEED_STD
    delta_mean_vs_td3bc = a_mean - TD3BC_PRIV_MEAN

    print(f"\nΔ mean (5s vs 3s): {delta_mean_vs_3seed:+.4f}  ({delta_mean_vs_3seed*100:+.1f}pp)")
    print(f"Δ std  (5s vs 3s): {delta_std_vs_3seed:+.4f}  (3→5 seeds 时 std 估计放宽是预期的)")
    print(f"Δ mean (ReBRAC priv 5s vs TD3BC priv 5s): {delta_mean_vs_td3bc:+.4f}  ({delta_mean_vs_td3bc*100:+.1f}pp)")


### 2.5 自动 verdict — 任务 A

In [ ]:
if not a_per_seed.empty:
    print("=" * 80)
    print("任务 A verdict — Phase 2 5-seed")
    print("-" * 80)

    cond_mean = a_mean > TD3BC_PRIV_MEAN
    cond_std  = a_std <= 0.10
    cond_consistency = abs(delta_mean_vs_3seed) <= 0.02
    cond_strict_std = a_std < TD3BC_PRIV_STD

    print(f"  [cond 1] 5-seed mean > TD3BC priv ({TD3BC_PRIV_MEAN:.3f})        : {a_mean:.4f}  → {'PASS' if cond_mean else 'FAIL'}")
    print(f"  [cond 2] 5-seed std  ≤ 0.10                          : {a_std:.4f}  → {'PASS' if cond_std else 'FAIL'}")
    print(f"  [cond 3] |Δ mean (5s − 3s)| ≤ 2pp                    : {delta_mean_vs_3seed*100:+.2f}pp  → {'PASS' if cond_consistency else 'WARN'}")
    print(f"  [cond 4] 5-seed std < TD3BC priv std ({TD3BC_PRIV_STD:.3f})    : {a_std:.4f}  → {'PASS（可声明严格更低）' if cond_strict_std else 'FAIL（不可声明严格更低，仅 claim 持平）'}")
    print("=" * 80)


## 3. 任务 B — critic LayerNorm-off probe（crosscomp-1000, 2 seeds）

### 3.1 scope / judgement

**目的**：[review §2.2.4](../docs/rebrac_mainline_review.md) — ReBRAC 原 paper 强调 critic LayerNorm 比 dual penalty 更关键。当前 winner 同时含 LN-on + dual penalty，未拆开过。审稿人会问 "提升其实是 LayerNorm 的功劳，dual penalty 是配角"。本 probe 把 LN 关掉，看 mean 与 mean_target_q 是否退化。

**实验 scope**：

| 轴 | 配置 | 说明 |
|---|---|---|
| β1 / β2 | `4.0 / 2.0` | Stage C finalist 不变 |
| dataset | `crosscomp-1000` | 与 Stage C / Stage E (a) 同 dataset，方便对照 |
| seeds | `42 43` | 2-seed cheap probe（与 worldcomp probe 同口径） |
| **critic_layernorm** | **`off`** | 关键操纵 |
| actor_layernorm | `off` | 与 winner 一致（默认 off） |
| TRAIN_EPOCHS | `64` | 与全线一致 |

**预算**：~2h L4。

**对照基线**：

| 协议 | dataset | mean | std | 来源 |
|---|---|---|---|---|
| ReBRAC `(β1=4.0, β2=2.0, LN=on)` | crosscomp-1000 | 0.902 | 0.021 | Stage C（5 seeds） |
| ReBRAC `(β1=4.0, β2=0, LN=on)` | crosscomp-1000 | 0.878 | 0.090 | Stage E (a)（5 seeds） |
| **ReBRAC `(β1=4.0, β2=2.0, LN=off)`** | crosscomp-1000 | ?? | ?? | 本实验（2 seeds） |

**判定区间**：

| 情形 | mean_test_success | mean_target_q 漂移 | 解读 |
|---|---|---|---|
| **A** LN 不关键 | 在 Stage C ± 2pp 内（0.882 ~ 0.922） | < +5 absolute units | LN 在本任务上不必要；与 ReBRAC 原 paper 不一致，本身是有趣 finding |
| **B** LN 关键 | 掉 ≥ 5pp | ≥ +5 absolute units | LN 是必要 component；dual penalty 与 LN 是独立贡献 |
| **C** LN 关键且 dual 退化 | 掉远超 5pp 且 std 飙升 | 大漂移 | dual penalty 在 LN-off 下失效；论文需重写归因 |

任一非 A 情形都能堵审稿人 "提升来自 LN" 的论断（dual penalty 即使 LN-on 仍贡献 mean=Stage C 的 -2.4pp 缓冲；LN 关掉若没崩，反而说明 dual penalty 是主导）。

### 3.2 生成一次性 sibling driver（注入 `--no-critic-layernorm`）

`scripts/run_offline_rebrac_screen.sh` 第 324 行硬编码 `--critic-layernorm`。直接 sed 复制一份并替换，不污染原 driver。

In [ ]:
import shutil
from pathlib import Path

src = Path("scripts/run_offline_rebrac_screen.sh")
dst = Path("scripts/run_offline_rebrac_critic_ln_off.sh")
shutil.copy2(src, dst)

# argparse BooleanOptionalAction 是左→右覆盖，把 --critic-layernorm 直接换成 --no-critic-layernorm 最干净
text = dst.read_text(encoding="utf-8")
patched = text.replace("--critic-layernorm \\", "--no-critic-layernorm \\", 1)
assert patched != text, "patch failed: --critic-layernorm not found"
dst.write_text(patched, encoding="utf-8")
dst.chmod(0o755)

# 显示 patch 后的两行（注入位置 ±5 行）
lines = patched.splitlines()
hit = next(i for i, ln in enumerate(lines) if "--no-critic-layernorm" in ln)
print(f"[patched] {dst}")
for i in range(max(0, hit - 2), min(len(lines), hit + 3)):
    marker = " >>>" if i == hit else "    "
    print(f"{marker} {i+1:>4}: {lines[i]}")


### 3.3 driver 配置 + 跑 train → validate → test → summarize

In [ ]:
# —— β1=4.0, β2=2.0（与 winner 一致，仅 LN 改变）——
os.environ["ACTOR_PENALTY_COEFS"]  = "4.0"
os.environ["CRITIC_PENALTY_COEFS"] = "2.0"

# —— 协议（crosscomp-1000，与 Stage C 主对照点对齐）——
os.environ["DATASET_POLICY"]          = "crosscomp"
os.environ["DATASET_EPISODES"]        = "1000"
os.environ["SEEDS"]                   = "42 43"
os.environ["TRAIN_EPOCHS"]            = "64"
os.environ["CHECKPOINT_EVERY_EPOCHS"] = "8"
os.environ["VAL_MANIFEST_EPISODES"]   = "40"
os.environ["TEST_MANIFEST_EPISODES"]  = "100"

# —— 输出根目录（与 Stage B/C/D/E 全部隔离）——
os.environ["CHECKPOINT_ROOT"] = "checkpoints/offline/rebrac/critic_ln_off"
os.environ["RESULTS_ROOT"]    = "results/offline/rebrac/critic_ln_off"
os.environ["SUMMARY_ROOT"]    = "results/offline/rebrac/critic_ln_off/summaries"

# Manifest 复用 Stage C 的 benchmarks/offline_rebrac_screen/{val_40,test_100}/
# 离线数据复用 offline_data/crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/


In [ ]:
# 全流程（用 sibling driver）
for mode in ["manifests", "collect", "train", "validate", "test", "summarize"]:
    os.environ["MODE"] = mode
    !bash scripts/run_offline_rebrac_critic_ln_off.sh


### 3.4 vs Stage C / Stage E (a) 三向对比 + mean_target_q 诊断

In [ ]:
B_RESULTS_ROOT = Path("results/offline/rebrac/critic_ln_off")
B_DATASET      = "crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000"
B_PAIR         = "actorb_4p0__criticb_2p0"
B_SEEDS        = os.environ["SEEDS"].split()

b_per_seed = load_test_per_seed(B_RESULTS_ROOT, B_DATASET, B_PAIR, B_SEEDS)
print("[per-seed LN-off probe — crosscomp-1000, β1=4.0, β2=2.0, LN=off, 2 seeds]")
print(b_per_seed.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
print()

if not b_per_seed.empty:
    b_mean = b_per_seed["success_rate"].mean()
    b_std  = b_per_seed["success_rate"].std()
    print(f"[summary] mean = {b_mean:.4f}, std = {b_std:.4f}")

    # 加载 mean_target_q
    import csv
    overview = B_RESULTS_ROOT / "summaries" / "overview.csv"
    b_target_q = None
    if overview.exists():
        with overview.open(encoding="utf-8") as fp:
            for row in csv.DictReader(fp):
                if row["pair"] == B_PAIR and row["dataset"] == B_DATASET:
                    b_target_q = float(row["mean_target_q"])
                    break
    print(f"[overview] mean_target_q = {b_target_q if b_target_q is not None else 'NA'}")


In [ ]:
# 三向对比 (Stage C / Stage E (a) / 本实验)
STAGE_C_MEAN, STAGE_C_STD, STAGE_C_TARGET_Q = 0.902, 0.021, -8.25
STAGE_E_MEAN, STAGE_E_STD, STAGE_E_TARGET_Q = 0.878, 0.090, -0.13

if not b_per_seed.empty:
    print("=" * 92)
    print(f"{'Configuration (crosscomp-1000)':<58}{'mean':>8}{'std':>8}{'target_q':>10}{'n':>4}")
    print("-" * 92)
    print(f"{'Stage C  (β1=4.0, β2=2.0, LN=on)':<58}{STAGE_C_MEAN:>8.3f}{STAGE_C_STD:>8.3f}{STAGE_C_TARGET_Q:>10.2f}{5:>4}")
    print(f"{'Stage E (β1=4.0, β2=0,   LN=on)':<58}{STAGE_E_MEAN:>8.3f}{STAGE_E_STD:>8.3f}{STAGE_E_TARGET_Q:>10.2f}{5:>4}")
    tq_show = b_target_q if b_target_q is not None else float("nan")
    print(f"{'This    (β1=4.0, β2=2.0, LN=off)':<58}{b_mean:>8.4f}{b_std:>8.4f}{tq_show:>10.2f}{2:>4}")
    print("=" * 92)

    print()
    print(f"Δ mean (LN-off vs Stage C):    {b_mean - STAGE_C_MEAN:+.4f}  ({(b_mean - STAGE_C_MEAN)*100:+.1f}pp)")
    print(f"Δ mean (LN-off vs Stage E):    {b_mean - STAGE_E_MEAN:+.4f}  (是否比 β2=0 更差？)")
    if b_target_q is not None:
        print(f"Δ target_q (LN-off vs Stage C): {b_target_q - STAGE_C_TARGET_Q:+.2f} (absolute)")
        print(f"Δ target_q (LN-off vs Stage E): {b_target_q - STAGE_E_TARGET_Q:+.2f}  (absolute)")


### 3.5 自动 verdict — 任务 B

In [ ]:
if not b_per_seed.empty:
    delta_mean_b = b_mean - STAGE_C_MEAN
    delta_target_q_b = (b_target_q - STAGE_C_TARGET_Q) if b_target_q is not None else None

    print("=" * 80)
    print("任务 B verdict — critic LayerNorm-off probe")
    print("-" * 80)

    if abs(delta_mean_b) <= 0.02 and (delta_target_q_b is None or abs(delta_target_q_b) < 5.0):
        verdict = ("情形 A：LN 不关键\n"
                   "  → ReBRAC 原 paper 在本任务（水下 AUV wake nav）上不可直接迁移\n"
                   "  → dual penalty 是主要 component；本身是有趣 finding，可写进 discussion")
    elif delta_mean_b <= -0.05 or (delta_target_q_b is not None and delta_target_q_b >= 5.0):
        verdict = ("情形 B：LN 关键，dual penalty 仍独立贡献\n"
                   "  → dual penalty 与 LN 是两个独立 component；可堵审稿人 'LN-only' 论断")
    else:
        verdict = ("情形 C：LN 关键且 dual penalty 同时受影响\n"
                   "  → 论文归因需小心写：winner 的 stability 是 (LN + dual penalty) 协同")

    print(verdict)
    print("=" * 80)


## 4. 任务 C — Phase 1 ReBRAC dep vs TD3BC priv 统计检验

### 4.1 数据准备

[review §2.2.3](../docs/rebrac_mainline_review.md)：5 seeds × test=100，跨 seed std≈0.077，标准误约 0.077/√5 ≈ 0.034；TD3BC priv 标准误约 0.038。Δ = +0.6pp 远小于 1 个标准误，因此 main text 应改写为 "持平" 而非 "超过"。

本节用三种方法量化：

1. **Paired episode-level bootstrap**（10000 resamples）：每个 episode_id 跨 5 seeds 平均，得到 100 paired observations，按 episode_id 重抽样。
2. **Welch's t-test on seed-level means**（不假设方差相等）：5 vs 5 seed-level 均值。
3. **95% CI on gap closure**：deployable→teacher gap closure（53.0% vs 48.5%）的不确定性。

In [ ]:
# 加载 ReBRAC Phase 1 deployable 与 TD3BC privileged-critic 的 per-seed × 100-episode 数据
REBRAC_DEP_ROOT = Path("results/offline/rebrac/worldcomp_teacher_gap/deployable")
REBRAC_DATASET  = "worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000"
REBRAC_PAIR     = "actorb_4p0__criticb_2p0"
REBRAC_SEEDS    = ["42", "43", "44", "45", "46"]

TD3BC_PRIV_ROOT = Path("results/offline/td3bc/phase0c/worldcomp_teacher_gap/privileged_final/worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone")
TD3BC_ALPHA_TAG = "alpha_0p1"
TD3BC_SEEDS     = ["42", "43", "44", "45", "46"]


def load_per_episode(root: Path, dataset: str, pair: str, seeds: list[str], subdir: str = "test") -> dict[str, list[dict]]:
    out: dict[str, list[dict]] = {}
    for seed in seeds:
        path = root / dataset / pair / subdir / f"seed_{seed}.json"
        d = json.loads(path.read_text(encoding="utf-8"))
        out[seed] = d["eval_episode_results"]
    return out


def load_per_episode_td3bc(root: Path, alpha_tag: str, seeds: list[str]) -> dict[str, list[dict]]:
    out: dict[str, list[dict]] = {}
    for seed in seeds:
        path = root / "test_selected" / alpha_tag / f"seed_{seed}.json"
        d = json.loads(path.read_text(encoding="utf-8"))
        out[seed] = d["eval_episode_results"]
    return out


rebrac_dep_eps = load_per_episode(REBRAC_DEP_ROOT, REBRAC_DATASET, REBRAC_PAIR, REBRAC_SEEDS)
td3bc_priv_eps = load_per_episode_td3bc(TD3BC_PRIV_ROOT, TD3BC_ALPHA_TAG, TD3BC_SEEDS)

print(f"[ReBRAC dep]  loaded {len(rebrac_dep_eps)} seeds, {len(rebrac_dep_eps['42'])} episodes/seed")
print(f"[TD3BC priv] loaded {len(td3bc_priv_eps)} seeds, {len(td3bc_priv_eps['42'])} episodes/seed")

# sanity check: episode_id 列表应跨 protocol / seed 一致
ep_ids_rebrac = [r["episode_id"] for r in rebrac_dep_eps["42"]]
ep_ids_td3bc  = [r["episode_id"] for r in td3bc_priv_eps["42"]]
assert ep_ids_rebrac == ep_ids_td3bc, "manifest 不一致；不能 paired bootstrap"
print(f"[sanity] episode_id 列表跨 protocol 一致：{len(ep_ids_rebrac)} eps / {ep_ids_rebrac[0]} ... {ep_ids_rebrac[-1]}")


### 4.2 Paired episode-level bootstrap

In [ ]:
import numpy as np

rng = np.random.default_rng(0)


def per_episode_mean_success(eps_per_seed: dict[str, list[dict]]) -> np.ndarray:
    """对每个 episode_id 跨 seeds 取 success indicator 的平均。返回形状 (n_episodes,) 的数组。"""
    seeds = sorted(eps_per_seed.keys())
    n_eps = len(eps_per_seed[seeds[0]])
    succ = np.zeros((len(seeds), n_eps), dtype=np.float64)
    for i, s in enumerate(seeds):
        for j, ep in enumerate(eps_per_seed[s]):
            succ[i, j] = 1.0 if ep["success"] else 0.0
    return succ.mean(axis=0)  # 跨 seeds 平均


rebrac_per_ep = per_episode_mean_success(rebrac_dep_eps)
td3bc_per_ep  = per_episode_mean_success(td3bc_priv_eps)

print(f"per-episode mean success (跨 5 seeds):")
print(f"  ReBRAC dep:  {rebrac_per_ep.mean():.4f}  (= 5-seed × 100-ep grand mean)")
print(f"  TD3BC priv:  {td3bc_per_ep.mean():.4f}")
print(f"  Δ (ReBRAC − TD3BC): {(rebrac_per_ep - td3bc_per_ep).mean():+.4f}")
print()

# Paired bootstrap by episode_id
N_BOOT = 10000
n_eps  = len(rebrac_per_ep)
deltas = np.zeros(N_BOOT, dtype=np.float64)
for b in range(N_BOOT):
    idx = rng.integers(0, n_eps, size=n_eps)
    deltas[b] = (rebrac_per_ep[idx] - td3bc_per_ep[idx]).mean()

ci_lo, ci_hi = np.percentile(deltas, [2.5, 97.5])
ci_99 = np.percentile(deltas, [0.5, 99.5])
p_two_sided = 2 * min((deltas <= 0).mean(), (deltas >= 0).mean())

print(f"[paired episode-level bootstrap, {N_BOOT} resamples]")
print(f"  point estimate Δ = {(rebrac_per_ep - td3bc_per_ep).mean():+.4f}")
print(f"  95% CI on Δ      = [{ci_lo:+.4f}, {ci_hi:+.4f}]")
print(f"  99% CI on Δ      = [{ci_99[0]:+.4f}, {ci_99[1]:+.4f}]")
print(f"  bootstrap p (two-sided, H0: Δ=0) ≈ {p_two_sided:.4f}")


### 4.3 Welch's t-test on seed-level means

In [ ]:
from scipy import stats

rebrac_seed_means = np.array([
    np.mean([1.0 if e["success"] else 0.0 for e in rebrac_dep_eps[s]])
    for s in REBRAC_SEEDS
])
td3bc_seed_means = np.array([
    np.mean([1.0 if e["success"] else 0.0 for e in td3bc_priv_eps[s]])
    for s in TD3BC_SEEDS
])

print(f"[seed-level means]")
for seed, rm, tm in zip(REBRAC_SEEDS, rebrac_seed_means, td3bc_seed_means):
    print(f"  seed {seed}: ReBRAC dep = {rm:.3f}   TD3BC priv = {tm:.3f}   Δ = {rm-tm:+.3f}")
print()
print(f"  ReBRAC dep:  mean = {rebrac_seed_means.mean():.4f}, std = {rebrac_seed_means.std(ddof=1):.4f}")
print(f"  TD3BC priv:  mean = {td3bc_seed_means.mean():.4f}, std = {td3bc_seed_means.std(ddof=1):.4f}")
print(f"  Δ mean    :  {rebrac_seed_means.mean() - td3bc_seed_means.mean():+.4f}")
print()

# Welch's t-test (不假设方差相等)
welch = stats.ttest_ind(rebrac_seed_means, td3bc_seed_means, equal_var=False)
print(f"[Welch's t-test on 5 seed-level means]")
print(f"  t = {welch.statistic:.3f}")
print(f"  p (two-sided) = {welch.pvalue:.4f}")
print(f"  → {'reject H0 (有显著差异)' if welch.pvalue < 0.05 else 'fail to reject H0 (持平，无显著差异)'}")


### 4.4 95% CI on gap closure

In [ ]:
# gap closure 定义：(method_mean - deployable_baseline) / (teacher - deployable_baseline)
# Phase 1 deployable / TD3BC deployable / teacher 的 mean 直接来自 report
TEACHER_MEAN = 0.990                    # worldcomp teacher
TD3BC_DEP_MEAN = 0.858                  # TD3BC deployable formal 5-seed mean
TD3BC_DEP_STD  = 0.080                  # TD3BC deployable std

# bootstrap on seed-level means: ReBRAC dep / TD3BC priv 各 5 seeds → gap closure 的不确定性
def gap_closure(method_mean: float, baseline: float = TD3BC_DEP_MEAN, teacher: float = TEACHER_MEAN) -> float:
    return (method_mean - baseline) / (teacher - baseline)

rebrac_gap = gap_closure(rebrac_seed_means.mean())
td3bc_gap  = gap_closure(td3bc_seed_means.mean())
print(f"point estimates:")
print(f"  ReBRAC dep gap closure  = {rebrac_gap*100:5.1f}%")
print(f"  TD3BC priv gap closure  = {td3bc_gap*100:5.1f}%")
print(f"  Δ                       = {(rebrac_gap - td3bc_gap)*100:+5.1f}pp")
print()

# bootstrap CI: 重抽样 5 seeds with replacement
N_BOOT_GAP = 10000
gap_diffs = np.zeros(N_BOOT_GAP)
for b in range(N_BOOT_GAP):
    idx = rng.integers(0, 5, size=5)
    rebrac_b = gap_closure(rebrac_seed_means[idx].mean())
    td3bc_b  = gap_closure(td3bc_seed_means[idx].mean())
    gap_diffs[b] = rebrac_b - td3bc_b

gap_ci = np.percentile(gap_diffs, [2.5, 97.5])
print(f"[bootstrap CI on gap closure Δ, {N_BOOT_GAP} resamples, seed-level resampling]")
print(f"  95% CI on Δ gap closure = [{gap_ci[0]*100:+.2f}pp, {gap_ci[1]*100:+.2f}pp]")
print(f"  → 由于 5 seeds 极少，bootstrap CI 较宽；'+4.5pp' 点估计不应在 main text 单独引用")


### 4.5 写入 docs/rebrac_statistical_test_followup.md

In [ ]:
STATS_MD = Path("docs/rebrac_statistical_test_followup.md")

header = f"""# ReBRAC Phase 1 deployable vs TD3BC privileged-critic — Statistical test follow-up

> 文档版本：rev.1（由 `notebooks/rebrac_paper_followup.ipynb` §4 自动生成）
> 配套：[docs/rebrac_mainline_review.md §2.2.3 / §3.1.C](./rebrac_mainline_review.md)
> 用途：paper drafting 时的 main results 表注脚 / discussion 引用。

## 1. Sample-level 数据

| Protocol | 来源 | n_seeds | n_eps/seed | seed-level mean | seed-level std |
|---|---|---:|---:|---:|---:|
| ReBRAC deployable Phase 1 | `results/offline/rebrac/worldcomp_teacher_gap/deployable/.../actorb_4p0__criticb_2p0/test/seed_*.json` | {len(REBRAC_SEEDS)} | {len(rebrac_dep_eps['42'])} | {rebrac_seed_means.mean():.4f} | {rebrac_seed_means.std(ddof=1):.4f} |
| TD3BC privileged-critic | `results/offline/td3bc/phase0c/worldcomp_teacher_gap/privileged_final/.../test_selected/alpha_0p1/seed_*.json` | {len(TD3BC_SEEDS)} | {len(td3bc_priv_eps['42'])} | {td3bc_seed_means.mean():.4f} | {td3bc_seed_means.std(ddof=1):.4f} |

Per-seed success rate（成功率 / 100 episodes）：

| seed | ReBRAC dep | TD3BC priv | Δ |
|---|---:|---:|---:|
"""

per_seed_rows = "".join(
    f"| {s} | {rm:.3f} | {tm:.3f} | {rm - tm:+.3f} |\n"
    for s, rm, tm in zip(REBRAC_SEEDS, rebrac_seed_means, td3bc_seed_means)
)

body = f"""
## 2. Paired episode-level bootstrap（10000 resamples）

按 100 个 episode_id 重抽样（跨 5 seeds 取每个 episode 的成功率均值，再 bootstrap）：

- point estimate Δ = **{(rebrac_per_ep - td3bc_per_ep).mean():+.4f}**
- 95% CI on Δ = [{ci_lo:+.4f}, {ci_hi:+.4f}]
- 99% CI on Δ = [{ci_99[0]:+.4f}, {ci_99[1]:+.4f}]
- bootstrap p (two-sided, H0: Δ=0) ≈ **{p_two_sided:.4f}**

## 3. Welch's t-test（5 vs 5 seed-level means）

- t = **{welch.statistic:.3f}**
- p (two-sided) = **{welch.pvalue:.4f}**
- 结论：**{'reject H0' if welch.pvalue < 0.05 else 'fail to reject H0（持平）'}**

## 4. Gap closure 95% CI（seed-level bootstrap）

deployable→teacher gap closure 定义 `(method - TD3BC_dep) / (teacher - TD3BC_dep)`，TD3BC_dep = {TD3BC_DEP_MEAN}, teacher = {TEACHER_MEAN}：

- ReBRAC dep gap closure point = **{rebrac_gap * 100:.1f}%**
- TD3BC priv gap closure point = **{td3bc_gap * 100:.1f}%**
- Δ point = {(rebrac_gap - td3bc_gap) * 100:+.1f}pp
- 95% CI on Δ = [{gap_ci[0] * 100:+.2f}pp, {gap_ci[1] * 100:+.2f}pp]

## 5. Paper 写作建议（堵 review §2.2.3）

- main text 的 ReBRAC dep vs TD3BC priv 结论改为 **"持平（statistically not different）"**，附 Welch's p-value；
- main results 表注脚写入 paired bootstrap CI 与 t-test p-value；
- gap closure +{(rebrac_gap - td3bc_gap) * 100:.1f}pp 仅在 discussion 中作为 directional 报告，不在 abstract / conclusion 中作为强 claim；
- 5 seeds 是 RL benchmark 的常见上限，但应在 limitations 显式声明 underpowered。
"""

content = header + per_seed_rows + body

STATS_MD.parent.mkdir(parents=True, exist_ok=True)
STATS_MD.write_text(content, encoding="utf-8")
print(f"[written] {STATS_MD}  ({len(content)} chars)")


## 5. 任务 D — Q-normalized 变体 method section 草稿

### 5.1 从 `auv_nav/rebrac.py` 提取实际 actor loss 实现

[review §2.2.1](../docs/rebrac_mainline_review.md)：当前实现是 Q-normalized dual-penalty TD3+BC，与 Tarasov et al. (2023) 原 ReBRAC 在 actor loss 形式上不同。需要在 paper method section 显式声明，以堵 "这不是 ReBRAC" 的论点。

In [ ]:
rebrac_src = Path("auv_nav/rebrac.py").read_text(encoding="utf-8")

# 抽出 actor loss 段（_actor_loss_terms 函数体）
import re
match = re.search(
    r"def _actor_loss_terms\(.*?\n(.*?)\n    def update",
    rebrac_src,
    flags=re.DOTALL,
)
assert match is not None, "未找到 _actor_loss_terms"
print("=== actor loss 实现（截取自 auv_nav/rebrac.py）===")
print(match.group(0).rstrip())


### 5.2 写入 docs/rebrac_method_section_draft.md

In [ ]:
METHOD_MD = Path("docs/rebrac_method_section_draft.md")

content = r"""# Method Section Draft — Q-normalized dual-penalty TD3+BC variant

> 文档版本：rev.1（由 `notebooks/rebrac_paper_followup.ipynb` §5 自动生成）
> 配套：[docs/rebrac_mainline_review.md §2.2.1 / §3.1.D](./rebrac_mainline_review.md)
> 用途：直接拷入 paper method section，并放入 implementation note。

## 1. Algorithm name and positioning

We refer to our offline RL algorithm as a **Q-normalized dual-penalty TD3+BC variant** (alias: ReBRAC-Q).
It implements the minimal recipe of ReBRAC (Tarasov et al., 2023)—dual BC penalty applied symmetrically to actor and critic plus critic LayerNorm—on top of the TD3+BC actor loss form (Fujimoto and Gu, 2021), in which the deterministic policy gradient is normalized by `|Q|.detach()`.

This variant is **not** identical to the original ReBRAC. The numerical value of `β_1` and `β_2` reported in this paper is therefore not directly comparable to the values in Tarasov et al. (2023).

## 2. Actor loss

For batch `(s, a)` drawn from the offline buffer:

- Let `π_θ(s)` be the deterministic actor and `Q_φ(s, a)` be the twin-critic minimum.
- Define the Q-normalization scalar (detached, per batch):

    λ = 1 / mean( |Q_φ(s, π_θ(s))| ).detach()
    (clamped at a numerical floor of 1e-6)

- The actor loss is:

    L_actor(θ) = − λ · E[ Q_φ(s, π_θ(s)) ] + β_1 · E[ ‖π_θ(s) − a‖² ]

The critic loss is the standard ReBRAC critic loss:

    L_critic(φ) = E[ (Q_φ(s, a) − y(s, a, r, s'))² ] + β_2 · E[ ‖a' − π_θ̄(s')‖² ]

where `a' ~ π_θ̄(s') + clipped noise` is the next-action target used for both bootstrapping and the critic-side BC penalty (TD3-style target smoothing applied to the actor target).

## 3. Difference from original ReBRAC

| Component | Original ReBRAC (Tarasov et al., 2023) | This work |
|---|---|---|
| Actor `Q` term scaling | `−E[Q]` (no normalization) | `−(1/|Q|.detach()) · E[Q]` |
| Actor BC penalty | `β_1 · E[‖π − a‖²]` | identical |
| Critic loss | TD3 + `β_2 · E[‖a' − π̄(s')‖²]` | identical |
| Critic LayerNorm | on by default | on (winner) |
| Policy update freq | every 2 critic updates | identical |
| Target smoothing | clipped Gaussian on next action | identical |

## 4. Difference from TD3+BC (Fujimoto and Gu, 2021)

Original TD3+BC actor loss:

    L_actor^{TD3+BC}(θ) = − λ_TD3+BC · E[ Q_φ(s, π_θ(s)) ] + E[ ‖π_θ(s) − a‖² ],
    with  λ_TD3+BC = α_TD3+BC / mean(|Q_φ|).detach()

Multiplying our actor loss by `1 / β_1` gives the equivalent normalized form:

    L_actor / β_1 = − (1 / (β_1 · |Q|.detach())) · E[ Q ] + E[ ‖π − a‖² ]

By matching coefficients, the BC anchoring strength of `β_1` in this work corresponds to TD3+BC `α ≈ 1 / β_1`. With `β_1 = 4.0` (winner), this matches `α ≈ 0.25`, which is also the α used by the TD3+BC baselines on `crosscomp` (phase0c).

The critical addition over TD3+BC is the **critic-side BC penalty** `β_2 · E[‖a' − π̄(s')‖²]`, which is the defining ReBRAC modification.

## 5. Implementation note (recommended for the paper)

> The actor loss in our implementation includes the TD3+BC-style Q normalization (`λ = 1/|Q|.detach()`) on the deterministic policy gradient term. The reported value `β_1 = 4.0` is the absolute coefficient of the BC penalty term and corresponds to the strongest anchoring level in our hyperparameter grid `{1.0, 2.0, 4.0}`. Numerical comparison of `β_1` and `β_2` against Tarasov et al. (2023) is not direct because of this scaling difference; we therefore restrict numerical hyperparameter comparisons to the TD3+BC baselines we trained ourselves under matched protocols.

## 6. Why this variant (justification, optional discussion bullet)

- Q normalization (TD3+BC) was retained because it removes a free hyperparameter (Q magnitude) that varies across `worldcomp` (mean Q ≈ +15) and `crosscomp` (mean Q ≈ −8), letting a single `β_1` work across datasets.
- Dual penalty (ReBRAC) was added because the critic-penalty-off probes (`β_2 = 0`) showed `mean_target_q` drift +46% / +98% across worldcomp and crosscomp, and the seed-44 outlier collapsed by −17pp in `crosscomp` without `β_2`.
- The combination lets us report a **single configuration** that is dataset-invariant across `worldcomp-1000`, `crosscomp-1000`, and `crosscomp-2000`, which would not be achievable with the original (un-normalized) ReBRAC actor loss without per-dataset retuning.
"""

METHOD_MD.parent.mkdir(parents=True, exist_ok=True)
METHOD_MD.write_text(content, encoding="utf-8")
print(f"[written] {METHOD_MD}  ({len(content)} chars)")
print()
print("Preview (first 60 lines):")
for line in content.splitlines()[:60]:
    print(line)


## 6. 报告写入清单

跑完上面所有 cell 后，按下表更新文档：

| 任务 | 写入位置 | 内容 |
|---|---|---|
| **A** | [docs/rebrac_experiment_report.md §7.12](../docs/rebrac_experiment_report.md) | 把 Phase 2 升级到 5-seed 数字（mean / std / Δ vs 3-seed / Δ vs TD3BC priv），追加严格 std 对比段落；§9 limitations 删去 "Phase 2 仅 3 seeds" 一条 |
| **A** | [docs/rebrac_experiment_plan.md §6.5.3](../docs/rebrac_experiment_plan.md) | Phase 2 标注 `【已扩展到 5-seed】`；rev 升 1 |
| **B** | [docs/rebrac_experiment_report.md §7.14](../docs/rebrac_experiment_report.md) | 新增 §7.14.x「critic LayerNorm-off probe」小节；填入 mean / std / mean_target_q / verdict |
| **B** | [docs/rebrac_mainline_review.md §2.2.4](../docs/rebrac_mainline_review.md) | 把 verdict 替换原文 "建议处理" 段落 |
| **C** | [docs/rebrac_statistical_test_followup.md](../docs/rebrac_statistical_test_followup.md) | 已由 §4.5 自动生成；paper drafting 时直接引用 |
| **C** | [docs/rebrac_mainline_review.md §2.2.3](../docs/rebrac_mainline_review.md) | 把 "建议处理" 段落改为 "已在 followup notebook §4 完成，详见 docs/rebrac_statistical_test_followup.md" |
| **D** | [docs/rebrac_method_section_draft.md](../docs/rebrac_method_section_draft.md) | 已由 §5.2 自动生成；paper method section 直接拷入 |
| **D** | [docs/rebrac_mainline_review.md §2.2.1](../docs/rebrac_mainline_review.md) | 把 "建议处理" 段落改为 "已在 followup notebook §5 完成，详见 docs/rebrac_method_section_draft.md" |

完成后执行：
```bash
git add docs/rebrac_statistical_test_followup.md docs/rebrac_method_section_draft.md \
        notebooks/rebrac_paper_followup.ipynb \
        scripts/run_offline_rebrac_critic_ln_off.sh \
        docs/rebrac_experiment_report.md docs/rebrac_experiment_plan.md docs/rebrac_mainline_review.md
git commit -m "docs(rebrac): paper-readiness follow-up (4 必做项)"
```

之后即可启动 paper drafting，不必再回到 ReBRAC 主线实验。